# Comparing a swept set of I2SB regressors

Loads every run under a sweep directory (`slurm/i2sb_tau_sweep.sbatch` or
`i2sb_beta_sweep.sbatch` -- the layout is the same) and scores them **on one shared batch** with
three diagnostics that answer different questions:

| | what it does | what it tells you |
|---|---|---|
| **1. One-shot** | set $x_t = x_1$, one network call at the $t{=}1$ end | how good the regressor is as a plain feed-forward synthesizer, with no sampling at all |
| **2. Full reconstruction** | the whole `nfe`-step reverse sampler | what you actually ship |
| **3. Per-step predictions** | $\hat x_0$ at a grid of bridge steps | where along the bridge each net is strong or weak |

Reading them together is the point. **(2) − (1)** is what the reverse sampling *buys you* over a
single pass; if it's ~0 the bridge is doing no work and you have an expensive regressor. And in
(3) we plot two curves that look similar but are not:

* **teacher-forced** -- $x_t$ built from the TRUE $(x_0,x_1)$ via `forward_sample`. This is exactly
  the training objective at each $t$, so it measures the regressor in isolation.
* **sampler trajectory** -- $\hat x_0$ logged from inside the reverse loop, where $x_t$ is whatever
  the sampler has produced so far. Errors compound here.

The gap between them is error accumulation in the sampler, which no single-pass metric can see.

Each run brings its **own schedule** (its own `tau`/`beta_max`), so every net is evaluated on the
bridge it was trained for. Only `nfe` is held fixed across runs, so the comparison is at equal cost.

In [ ]:
import os, json, glob, gc
import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline

# os.chdir('/scratch/ee2178/ImMAP')   # <-- EDIT to your repo root (ckpt paths in the configs
                                      #     are relative to it)

import datasets                                  # registers the loaders
from datasets.registry import build_loader
from training.common import load_model
from sb.base import build_schedule, n_steps, forward_sample, forward_std, predict_x0
from sb.i2sb import i2sb_sample

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# The only line to change to switch sweeps -- everything below reads the runs' own configs, so
# the tau sweep (kind="brownian") and the beta_max sweep (kind="i2sb") both work as-is.
SWEEP_ROOT = "trained_nets/brats/I2SB_beta_max_sweep"   # <-- or .../I2SB_tau_sweep
NFE        = 20        # held FIXED across runs so the comparison is at equal cost
N_SAMPLES  = 1         # >1 re-samples the stochastic bridge and also reports diversity
BATCH      = 8
CROP       = 192       # fixed center crop, so every run sees identical pixels
SEED       = 0
LOG_STEPS  = 6         # how many intermediate pred_x0 to log from inside the sampler

# ---- display window (fixed, not per-image) ----
# Panels are shown on [VMIN, VMAX] rather than per-image percentiles. Comparing a SWEEP is the
# case where this matters most: per-image windowing renormalizes every arm to its own contents,
# so an arm that systematically under-predicts contrast gets stretched back to full range and
# looks the same as one that got it right. Signed residuals use the symmetric [-VMAX, VMAX].
VMIN, VMAX = -1.0, 1.0

# ---- discover runs: any subdirectory carrying both a saved config and a checkpoint ----
runs = []
for cfg_path in sorted(glob.glob(os.path.join(SWEEP_ROOT, "*", "config.json"))):
    run_dir = os.path.dirname(cfg_path)
    if not os.path.exists(os.path.join(run_dir, "net.ckpt")):
        print(f"[skip] no net.ckpt in {run_dir}")
        continue
    with open(cfg_path) as f:
        runs.append({"dir": run_dir, "cfg_path": cfg_path, "cfg": json.load(f)})
if not runs:
    raise RuntimeError(f"no finished runs under {SWEEP_ROOT} "
                       f"(expected <run>/config.json + <run>/net.ckpt)")

# ---- which knob was actually swept? read it from the configs rather than the folder names ----
def sweep_key(runs):
    for k in ("tau", "beta_max", "n_points", "posterior", "deterministic", "kind"):
        vals = [r["cfg"]["i2sb"].get(k) for r in runs]
        if len(set(map(str, vals))) > 1:
            return k
    return None

KEY = sweep_key(runs)
for r in runs:
    r["label"] = (f"{KEY}={r['cfg']['i2sb'][KEY]:g}" if KEY and
                  isinstance(r["cfg"]["i2sb"].get(KEY), (int, float))
                  else (f"{KEY}={r['cfg']['i2sb'][KEY]}" if KEY else os.path.basename(r["dir"])))
    r["sort"] = r["cfg"]["i2sb"].get(KEY) if KEY else r["dir"]
try:
    runs.sort(key=lambda r: float(r["sort"]))
except (TypeError, ValueError):
    runs.sort(key=lambda r: str(r["sort"]))

print(f"{len(runs)} run(s) under {SWEEP_ROOT}; swept knob: {KEY or '(none detected)'}\n")
# "-" rather than nan for a key the schedule genuinely does not have: kind="i2sb" configs carry
# no `tau` at all, and nan in a results table reads like a failure instead of "not applicable".
fmt = lambda v, w: (f"{v:>{w}.3f}" if isinstance(v, (int, float)) else f"{'-':>{w}}")
print(f"{'label':<18} {'kind':<10} {'tau':>7} {'beta_max':>9} {'model':<10} {'C':>3}  dir")
for r in runs:
    ic, mc = r["cfg"]["i2sb"], r["cfg"]["model"]
    print(f"{r['label']:<18} {ic.get('kind',''):<10} {fmt(ic.get('tau'), 7)} "
          f"{fmt(ic.get('beta_max'), 9)} {mc['type']:<10} "
          f"{mc['params'].get('C','?'):>3}  {os.path.basename(r['dir'])}")

## One shared batch

Every run is scored on the *same* pixels: fixed center crop, no flips, fixed seed. The loader
config is taken from the first run so `x0_idx` / `x1_idx` / `cond_idx` / `scales` match what these
nets were trained on -- **if the runs disagree on those, the comparison is not apples-to-apples**
and the cell says so.

`et_mask: true` also pulls the enhancing-tumor mask, so we can score the region that actually
distinguishes T1ce from T1 (whole-brain PSNR barely moves on ~0.3% of voxels).

In [ ]:
base_data = dict(runs[0]["cfg"]["data"]["val"])

# a mismatch here would silently make the comparison meaningless
for field in ("x0_idx", "x1_idx", "cond_idx", "scales", "image_key", "x1_source", "root"):
    vals = {str(r["cfg"]["data"]["val"].get(field)) for r in runs}
    if len(vals) > 1:
        print(f"[WARN] runs disagree on data.{field}: {vals} -- not an apples-to-apples comparison")

base_data.update(name="i2sb", center_crop=CROP, random_flips=False, num_workers=0,
                 batch_size=BATCH, et_mask=True)
base_data.pop("crop_size", None)          # center_crop instead, so the batch is deterministic

torch.manual_seed(SEED); np.random.seed(SEED)
try:
    loader = build_loader(base_data, shuffle=True, drop_last=False)
    batch = next(iter(loader))
except (KeyError, RuntimeError) as e:      # h5 without 'et' -> fall back to brain mask only
    print(f"[warn] et_mask failed ({type(e).__name__}: {str(e)[:70]}...); continuing without ET")
    base_data["et_mask"] = False
    torch.manual_seed(SEED)
    loader = build_loader(base_data, shuffle=True, drop_last=False)
    batch = next(iter(loader))

x0, x1, cond, mask = (t.to(device) for t in batch[:4])
et = batch[4].to(device) if len(batch) > 4 else torch.zeros_like(mask)
cond = None if cond.shape[1] == 0 else cond

print(f"batch x0{tuple(x0.shape)} x1{tuple(x1.shape)} "
      f"cond{tuple(cond.shape) if cond is not None else None} mask{tuple(mask.shape)}")
print(f"brain voxels {int(mask.sum()):,}  ET voxels {int(et.sum()):,} "
      f"({float(et.sum()/mask.sum().clamp(min=1)):.3%} of brain)")

## Metrics

PSNR uses an **explicit** `data_range` measured from the ground truth over the brain (p99.5 − p0.5)
and printed below -- these images are z-scored, not in [0,1], so an implicit `data_range = 1` would
silently misstate every number. ET-PSNR is area-normalized over the region.

In [ ]:
from training.metrics import ssim as ssim_fn

DATA_RANGE = float(torch.quantile(x0[mask > 0.5].float(), 0.995)
                   - torch.quantile(x0[mask > 0.5].float(), 0.005))
print(f"data_range = {DATA_RANGE:.4f}  (p99.5 - p0.5 of x0 over the brain)")


def region_psnr(gt, pred, region):
    """Area-normalized PSNR inside a binary region (mask or ET)."""
    n = region.sum()
    if float(n) == 0:
        return float("nan")
    mse = float((region * (gt - pred) ** 2).sum() / n)
    return 10.0 * np.log10(DATA_RANGE ** 2 / max(mse, 1e-12))


def score(pred):
    """-> dict of the numbers we compare runs on."""
    return {
        "psnr": region_psnr(x0, pred, mask),
        "et_psnr": region_psnr(x0, pred, et),
        "ssim": float(ssim_fn(x0 * mask, pred * mask).mean()),
    }


# the floor every method must beat: answer the prior and do nothing
print("baseline (x_hat = x1):  " + "  ".join(f"{k}={v:.4f}" for k, v in score(x1).items()))

## Load, run all three diagnostics, free

One run at a time -- a sweep of GroupCDLs will not all fit in memory at once, so each net is loaded,
scored, and released before the next.

**Diagnostic 1 (one-shot)** is a single `predict_x0` at $x_t = x_1$ with $\sigma$ at the $t{=}1$ end
of that run's own schedule -- i.e. exactly the first call the reverse sampler makes, and nothing
more. **Diagnostic 2** runs the sampler. **Diagnostic 3a** is teacher-forced: $x_t$ built from the
true endpoints at a grid of steps.

In [ ]:
T_GRID = np.linspace(0.02, 0.98, 13)      # bridge positions for the teacher-forced sweep


def logged_positions(n, nfe, log_count):
    """The exact bridge positions reverse_sample's logged pred_x0 were PREDICTED at, ascending
    (= the order the returned array is in). Recomputed from sb.base's own space_indices rather
    than assumed uniform, so the trajectory panel shares an x-axis with the teacher-forced one.

    Note the off-by-one: reverse_sample walks pairs (n, n_prev), computes x0_hat AT n, but logs
    it when `n_prev` is a log step. So the position of a logged prediction is its `n` -- one
    checkpoint ABOVE the n_prev that triggered the log. Keying on n_prev instead would place the
    very first prediction (the one-shot, made at step interval-1) a checkpoint short of the prior
    end."""
    from sb.base import space_indices
    steps_all = space_indices(n, nfe + 1)                       # ascending checkpoints
    lc = min(len(steps_all) - 1, log_count)
    log_idx = sorted(set(space_indices(len(steps_all) - 1, lc)))    # indices of the logged n_prev
    return np.array([steps_all[i + 1] for i in log_idx], dtype=float) / n


@torch.no_grad()
def evaluate_run(r):
    cfg, ic = r["cfg"], r["cfg"]["i2sb"]
    net = load_model(r["cfg_path"], device=device)
    net.eval()
    if getattr(net, "attn_backend", None) == "flex":
        net.compile_flex()

    sched = build_schedule(kind=ic.get("kind", "brownian"), tau=ic.get("tau", 0.19),
                           n_points=ic.get("n_points", 1000),
                           beta_max=ic.get("beta_max", 0.3), device=device)
    n = n_steps(sched)
    tc = ic.get("target_channels", 1)
    det, post = ic.get("deterministic", False), ic.get("posterior", "ddpm")
    out = {"sched": sched, "n": n}

    # ---- 1. one-shot: x_t = x1 at the t=1 end, ONE network call ----
    b = x0.shape[0]
    step_end = torch.full((b,), n - 1, device=device, dtype=torch.long)
    sig_end = forward_std(sched, step_end, xdim=x0.shape[1:])
    out["oneshot"] = predict_x0(net, x1, sig_end, cond=cond, target_channels=tc)

    # ---- 2. full reconstruction (+ the sampler's own pred_x0 trajectory) ----
    recons, trajs = [], []
    for s in range(N_SAMPLES):
        torch.manual_seed(SEED + s)
        rec, _, px = i2sb_sample(net, x1, sched, cond=cond, nfe=NFE, deterministic=det,
                                 posterior=post, clip_denoise=ic.get("clip_denoise", False),
                                 target_channels=tc, log_count=LOG_STEPS, verbose=False)
        recons.append(rec); trajs.append(px)
    out["recon"] = torch.stack(recons).mean(0) if N_SAMPLES > 1 else recons[0]
    out["recon_std"] = (torch.stack(recons).std(0) if N_SAMPLES > 1
                        else torch.zeros_like(recons[0]))
    # reverse_sample returns pred_x0s NEWEST-FIRST, and "newest" is the last one computed, i.e.
    # the TARGET end. So index 0 is already t=0 and the array is in ascending bridge order --
    # do NOT flip it, or the trajectory panel comes out mirrored against the teacher-forced one.
    out["traj"] = trajs[0]
    out["traj_t"] = logged_positions(n, NFE, LOG_STEPS)

    # ---- 3a. teacher-forced single pass at a grid of bridge steps ----
    tf = []
    for tt in T_GRID:
        k = min(int(tt * n), n - 1)
        stp = torch.full((b,), k, device=device, dtype=torch.long)
        xt = forward_sample(sched, stp, x0, x1, deterministic=det)
        sig = forward_std(sched, stp, xdim=x0.shape[1:])
        tf.append(predict_x0(net, xt, sig, cond=cond, target_channels=tc))
    out["teacher"] = torch.stack(tf, dim=1)        # (B, len(T_GRID), 1, H, W)

    del net
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return out


results = {}
for r in runs:
    print(f"-- {r['label']} ...", flush=True)
    results[r["label"]] = evaluate_run(r)
print("done")

## Summary table

`gain` is full-reconstruction PSNR minus one-shot PSNR: **what the reverse sampling bought.**
Negative means the sampler is actively hurting, which is worth knowing before tuning anything else.

In [ ]:
rows = []
for r in runs:
    o = results[r["label"]]
    s1, s2 = score(o["oneshot"]), score(o["recon"])
    rows.append(dict(label=r["label"], one=s1["psnr"], full=s2["psnr"],
                     one_et=s1["et_psnr"], full_et=s2["et_psnr"], ssim=s2["ssim"],
                     gain=s2["psnr"] - s1["psnr"], gain_et=s2["et_psnr"] - s1["et_psnr"],
                     div=float(o["recon_std"][mask > 0.5].mean()) if N_SAMPLES > 1 else float("nan")))

hdr = (f"{'run':<18} {'1-shot':>8} {'full':>8} {'gain':>7} | "
       f"{'1shot ET':>9} {'full ET':>8} {'gainET':>7} | {'ssim':>6}")
print(hdr + (f" {'diversity':>10}" if N_SAMPLES > 1 else ""))
print("-" * (len(hdr) + (11 if N_SAMPLES > 1 else 0)))
for d in rows:
    line = (f"{d['label']:<18} {d['one']:>8.3f} {d['full']:>8.3f} {d['gain']:>+7.3f} | "
            f"{d['one_et']:>9.3f} {d['full_et']:>8.3f} {d['gain_et']:>+7.3f} | {d['ssim']:>6.4f}")
    print(line + (f" {d['div']:>10.4f}" if N_SAMPLES > 1 else ""))

best_full = max(rows, key=lambda d: d["full"])
best_et = max(rows, key=lambda d: d["full_et"])
print(f"\nbest full-recon PSNR : {best_full['label']} ({best_full['full']:.3f} dB)")
print(f"best full-recon ET   : {best_et['label']} ({best_et['full_et']:.3f} dB)")
print(f"NFE={NFE}, N_SAMPLES={N_SAMPLES}, batch={x0.shape[0]} slices -- one batch only; "
      f"widen it before trusting small differences.")

## 1 vs 2: does sampling help?

In [ ]:
lbl = [d["label"] for d in rows]
xs = np.arange(len(rows))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, (k1, k2, name) in zip(ax, [("one", "full", "whole brain"),
                                  ("one_et", "full_et", "enhancing tumor")]):
    a.plot(xs, [d[k1] for d in rows], "-o", label="one-shot ($x_t=x_1$)")
    a.plot(xs, [d[k2] for d in rows], "-s", label=f"full recon (nfe={NFE})")
    a.axhline(region_psnr(x0, x1, mask if name == "whole brain" else et),
              color="k", ls="--", lw=1, label="prior only ($x_1$)")
    a.set_xticks(xs); a.set_xticklabels(lbl, rotation=45, ha="right")
    a.set_ylabel("PSNR (dB)"); a.set_title(name); a.grid(alpha=.3); a.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 3: per-step predictions

**Left** -- teacher-forced: the regressor evaluated on a *correct* $x_t$ at each bridge position.
This is the training objective, so it isolates model quality; it should rise monotonically toward
$t=0$ as $x_t$ carries more of $x_0$.

**Right** -- the sampler's own $\hat x_0$ along the reverse trajectory. It starts at the same place
(the one-shot number, at $t=1$) but thereafter consumes its own output. **Where the right curve
falls below the left one at matching $t$, the sampler is accumulating error** -- that gap is
invisible to the single-pass validation `train_i2sb` logs.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
for r in runs:
    o = results[r["label"]]
    # teacher-forced, per t
    tf = [region_psnr(x0, o["teacher"][:, i], mask) for i in range(o["teacher"].shape[1])]
    ax[0].plot(T_GRID, tf, "-o", ms=3, label=r["label"])
    # sampler trajectory, at the EXACT positions reverse_sample logged (already ascending)
    tr, ts = o["traj"], o["traj_t"]
    assert tr.shape[1] == len(ts), "logged-position bookkeeping is out of sync with the trajectory"
    ax[1].plot(ts, [region_psnr(x0, tr[:, i].to(device), mask) for i in range(tr.shape[1])],
               "-s", ms=3, label=r["label"])

for a, ttl in zip(ax, ["teacher-forced (true $x_t$)", "sampler trajectory (own $x_t$)"]):
    a.axhline(region_psnr(x0, x1, mask), color="k", ls="--", lw=1, label="prior only")
    a.set_xlabel("bridge position  (0 = target end, 1 = prior end)")
    a.set_title(ttl); a.grid(alpha=.3); a.invert_xaxis()
ax[0].set_ylabel("brain PSNR (dB)"); ax[0].legend(fontsize=7)
plt.tight_layout(); plt.show()

## Images

One slice per column block: the prior, each run's one-shot and full reconstruction, and the truth.
Windowed jointly from the GT so the columns are directly comparable.

In [ ]:
SLICE = 0
cols = [("T1 prior", x1)] + \
       [(f"{r['label']}\none-shot", results[r["label"]]["oneshot"]) for r in runs] + \
       [(f"{r['label']}\nfull", results[r["label"]]["recon"]) for r in runs] + \
       [("T1ce GT", x0)]

lo, hi = VMIN, VMAX          # fixed window: every run in the sweep on the same greyscale

ncol = min(len(cols), 8)
nrow = int(np.ceil(len(cols) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(2.1 * ncol, 2.5 * nrow), squeeze=False)
for i, (name, img) in enumerate(cols):
    a = axes[i // ncol][i % ncol]
    a.imshow((img[SLICE, 0] * mask[SLICE, 0]).detach().cpu(), cmap="gray", vmin=lo, vmax=hi)
    a.set_title(name, fontsize=7)
    if name not in ("T1 prior", "T1ce GT"):
        a.set_xlabel(f"{region_psnr(x0, img, mask):.2f} dB", fontsize=7)
    a.set_xticks([]); a.set_yticks([])
for j in range(len(cols), nrow * ncol):
    axes[j // ncol][j % ncol].axis("off")
plt.tight_layout(); plt.show()

## Caveats

* **One batch.** Every number above comes from a single batch of `BATCH` slices. Differences of a
  few tenths of a dB are noise at that size -- widen `BATCH`, or loop the whole notebook over
  several batches, before ranking runs that finish close together.
* **PSNR ranks the wrong thing if you care about realism.** A stochastic bridge trades distortion
  for perceptual quality, so the run with the best PSNR is often the one injecting the least
  noise -- i.e. the most regressor-like. If the goal is realistic enhancement texture rather than
  minimum MSE, set `N_SAMPLES > 1` and read the diversity column alongside, or add LPIPS.
* **One-shot uses $\sigma$ at each run's own $t{=}1$ end**, which differs across the sweep by
  construction (for the Brownian schedule it is $2\tau$). That is deliberate -- each net is asked
  the question it was trained on -- but it does mean the one-shot column is not a fixed-$\sigma$
  comparison.

## Single-run eval

Pick one run and lay it out the way the other testbenches do: the three input contrasts, the
I2SB reconstruction, the T1ce ground truth, and the residual.

`PICK` takes an index into `runs`, a label substring, or a **path to any `config.json`** --
so this also works for a run trained outside this sweep, not just an arm of it. With `None` it
picks the best full-reconstruction PSNR.

The contrast columns are re-read from the h5 with `cond_idx = [0, 1, 3]` (stored order is
`flair=0, t1=1, t1ce=2, t2=3`) rather than taken from the shared `cond` tensor. `cond` holds only
what the run was *conditioned* on -- `[0, 3]` for `i2sb_paper.json`, empty for
`i2sb_gcdl_t1.json` -- so reading the panels off it would show different contrasts depending on
which run you picked, or nothing at all. Same root / crop / seed, so the shuffle lands on the same
slices; the cell asserts that rather than assuming it.

Windowing: **recon and GT share one window** so that pair is directly comparable, while each input
contrast gets its own -- the h5 is z-scored per contrast, and forcing T2/FLAIR into T1's window
washes them out.

In [ ]:
# ---- what to look at -------------------------------------------------------------------
PICK     = None      # None -> best full-recon PSNR. Also: an int index into `runs`, a label
                     # substring ("beta_max=0.3"), or a path to ANY run's config.json.
N_SHOW   = 4         # slices shown (rows), capped at the batch size
RESIDUAL = "error"   # "error"       -> recon - T1ce : what this run got wrong
                     # "enhancement" -> recon - T1   : the signal it was asked to synthesize
SHOW_ET  = True      # outline the enhancing-tumor mask on the recon / GT / residual panels
# ----------------------------------------------------------------------------------------

def _resolve(pick):
    if pick is None:
        return max(runs, key=lambda r: score(results[r["label"]]["recon"])["psnr"])
    if isinstance(pick, int):
        return runs[pick]
    if os.path.exists(pick):                       # any saved run, in this sweep or not
        with open(pick) as f:
            cfg = json.load(f)
        return {"dir": os.path.dirname(pick), "cfg_path": pick, "cfg": cfg,
                "label": os.path.basename(os.path.dirname(pick))}
    hit = [r for r in runs if pick in r["label"]]
    if len(hit) != 1:
        raise ValueError(f"PICK={pick!r} matched {[r['label'] for r in hit] or 'nothing'}; "
                         f"available: {[r['label'] for r in runs]}")
    return hit[0]

R, LBL = (lambda r: (r, r["label"]))(_resolve(PICK))
print(f"run : {LBL}\n      {R['cfg_path']}")

# ---- the three input contrasts ----------------------------------------------------------
_disp = dict(base_data); _disp["cond_idx"] = [0, 1, 3]      # flair, t1, t2
torch.manual_seed(SEED); np.random.seed(SEED)
_b = next(iter(build_loader(_disp, shuffle=True, drop_last=False)))
assert torch.allclose(_b[0].to(device), x0, atol=1e-5), \
    "display loader desynced from the shared batch -- the panels would be mislabelled"
_dc = _b[2].to(device)
FLAIR, T1, T2 = _dc[:, 0:1], _dc[:, 1:2], _dc[:, 2:3]
STORED = {0: FLAIR, 1: T1, 2: x0, 3: T2}                    # to rebuild any run's cond_idx

# ---- the reconstruction ------------------------------------------------------------------
if LBL in results:
    recon = results[LBL]["recon"]                            # already sampled above; reuse
else:
    ic, dv = R["cfg"]["i2sb"], R["cfg"]["data"]["val"]
    # the shared batch was built from runs[0]'s data block; if this run disagrees, x0/x1 here
    # are not the endpoints it trained on and the numbers below are not its numbers
    for f_ in ("x0_idx", "x1_idx", "x1_source", "scales", "image_key", "root"):
        if str(dv.get(f_)) != str(base_data.get(f_)):
            print(f"[WARN] data.{f_}: run has {dv.get(f_)!r}, shared batch has "
                  f"{base_data.get(f_)!r} -- not the endpoints this net trained on")
    ci = list(dv.get("cond_idx", []))
    sched_r = build_schedule(kind=ic.get("kind", "brownian"), tau=ic.get("tau", 0.19),
                             n_points=ic.get("n_points", 1000),
                             beta_max=ic.get("beta_max", 0.3), device=device)
    net = load_model(R["cfg_path"], device=device); net.eval()
    if getattr(net, "attn_backend", None) == "flex":
        net.compile_flex()
    torch.manual_seed(SEED)
    with torch.no_grad():
        recon, _, _ = i2sb_sample(net, x1, sched_r,
                                  cond=torch.cat([STORED[i] for i in ci], 1) if ci else None,
                                  nfe=NFE, deterministic=ic.get("deterministic", False),
                                  posterior=ic.get("posterior", "ddpm"),
                                  clip_denoise=ic.get("clip_denoise", False),
                                  target_channels=ic.get("target_channels", 1),
                                  log_count=1, verbose=False)
    del net; gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

_sr, _sb = score(recon), score(x1)
print(f"\nrecon : psnr={_sr['psnr']:7.3f}  et_psnr={_sr['et_psnr']:7.3f}  ssim={_sr['ssim']:.4f}")
print(f"prior : psnr={_sb['psnr']:7.3f}  et_psnr={_sb['et_psnr']:7.3f}  ssim={_sb['ssim']:.4f}"
      f"   <- x1, the do-nothing floor")

# ---- panel --------------------------------------------------------------------------------
nshow = min(N_SHOW, x0.shape[0])
if RESIDUAL == "error":
    res, res_ttl = recon - x0, "residual\n(recon - T1ce)"
elif RESIDUAL == "enhancement":
    res, res_ttl = recon - T1, "residual\n(recon - T1)"
else:
    raise ValueError(f"RESIDUAL={RESIDUAL!r} must be 'error' or 'enhancement'")

# One fixed window for every panel -- inputs, recon and GT alike. The med/MAD + scales=3
# normalization puts all four contrasts on a common scale, so there is no longer a reason to
# window each contrast separately, and a shared window is what makes the columns comparable.
_w = (VMIN, VMAX)
cols = [("T1",    T1,    _w, False),
        ("T2",    T2,    _w, False),
        ("FLAIR", FLAIR, _w, False),
        (f"I2SB recon\n{LBL} (nfe={NFE})", recon, _w, True),
        ("T1ce GT", x0, _w, False)]

rv = VMAX                                  # signed residual on the symmetric [-VMAX, VMAX]

def _outline(a, i, color="lime", lw=0.6):
    if not SHOW_ET:
        return
    e = et[i, 0].detach().cpu().numpy()
    if e.max() > 0:
        a.contour(e, levels=[0.5], colors=color, linewidths=lw)

fig, ax = plt.subplots(nshow, len(cols) + 1, squeeze=False,
                       figsize=(2.25 * (len(cols) + 1), 2.6 * nshow))
for i in range(nshow):
    for j, (name, img, (lo, hi), annot) in enumerate(cols):
        a = ax[i, j]
        a.imshow((img[i, 0] * mask[i, 0]).detach().cpu(), cmap="gray", vmin=lo, vmax=hi)
        if i == 0:
            a.set_title(name, fontsize=8)
        if annot:
            a.set_xlabel(f"{region_psnr(x0[i:i+1], img[i:i+1], mask[i:i+1]):.2f} dB | "
                         f"ET {region_psnr(x0[i:i+1], img[i:i+1], et[i:i+1]):.2f}", fontsize=7)
        if name.startswith(("I2SB", "T1ce")):
            _outline(a, i)
        a.set_xticks([]); a.set_yticks([])
    a = ax[i, -1]
    im_ = a.imshow((res[i, 0] * mask[i, 0]).detach().cpu(), cmap="bwr", vmin=-rv, vmax=rv)
    if i == 0:
        a.set_title(res_ttl, fontsize=8)
    _outline(a, i, color="k", lw=0.5)
    a.set_xticks([]); a.set_yticks([])

fig.colorbar(im_, ax=ax, shrink=0.6, label=f"residual (fixed, +/-{rv:g}, white = 0)")
plt.suptitle(f"{LBL}   nfe={NFE}   data_range={DATA_RANGE:.3f}", y=1.005, fontsize=10)
plt.show()